# 03 — Microsoft Foundry Setup

**Stage 1 — Baselines** · infrastructure sub-block (transversal cloud provider).

Configure Microsoft Foundry as the single entry point for all cloud models in the project: embeddings (Stage 1), reranking (Stage 2), LLMs for Contextual Retrieval (Stage 2-3).

Descriptive memory in Notion: [Stage 1 — Microsoft Foundry Setup](https://www.notion.so/35a6ad30ca11811d96ebf4a9d7dde20b).
Credentials setup: [`docs/foundry_setup.md`](../docs/foundry_setup.md).

---

## Sub-block index

1. What is Microsoft Foundry and why we use it transversally?
2. Account setup + credentials
3. SDK installation
4. Connection smoke test
5. Embeddings smoke test on 1 chunk of the FinanceBench corpus
6. Reusable `FoundryEmbedder` wrapper
7. Basic cost tracking

## Sub-block 1 — What is Microsoft Foundry and why we use it?

Microsoft Foundry is a unified Azure platform-as-a-service offering for enterprise AI operations, model builders, and application development. This foundation combines production-grade infrastructure with friendly interfaces, enabling developers to focus on building applications rather than managing infrastructure.

Microsoft Foundry unifies **agents, models, and tools** under a single management grouping with built-in enterprise-readiness capabilities including tracing, monitoring, evaluations, and customizable enterprise setup configurations. The platform provides streamlined management through unified role-based access control (RBAC), networking, and policies under one Azure resource provider namespace.

## Sub-block 2 — Account setup + credentials

See [`docs/foundry_setup.md`](../docs/foundry_setup.md) to get endpoint + key + deployment name.

## Sub-block 3 — SDK installation

There are **3 different SDKs** to talk to Microsoft Foundry from Python. Each one points to a different service of the same Foundry resource:

| SDK | Endpoint | What it's for |
|-----|----------|---------------|
| `azure-ai-projects` | `services.ai.azure.com/api/projects/<name>` | Agents, evaluations, fine-tuning, project management |
| `azure-ai-inference` | `services.ai.azure.com/models` | Foundry Models multi-model (Cohere, Llama, BGE...) — ⚠️ **deprecated, retires Aug 26 2026** |
| **`openai`** | **`openai.azure.com/openai/v1/`** | **Embeddings + Chat + Image — full OpenAI API surface** ✅ |

### Decision: we use `openai`

Technical reasons (all confirmed in official Microsoft docs):

1. **Microsoft itself recommends it** for embeddings — verbatim quote:
   > *"Use the OpenAI SDK endpoint for generating embeddings. The project endpoint used by the Foundry SDK doesn't currently route embedding requests."*
2. **`azure-ai-inference` is deprecated** — retires on **August 26, 2026**. Microsoft asks to migrate to the `openai` SDK.
3. **`azure-ai-projects` doesn't route embeddings** — it's only for agents/evaluations.
4. **Portability** — if at some point we want to migrate from Azure OpenAI to OpenAI directly, we change 1 line (`base_url`). Same code, different provider.

### Installation

At the project level, only once (NOT reinstalled from the notebook):

```bash
uv add openai
```

That added it to `pyproject.toml` + `uv.lock`. Any dev who clones the repo + runs `uv sync` already has it.

The cell below is just **verification** that the SDK is importable in the current kernel.

In [ ]:
# Validate that the OpenAI SDK is installed and importable in the current kernel
import openai
from openai import OpenAI

print(f"✅ openai {openai.__version__} installed")
print(f"   OpenAI client: importable")

## Sub-block 4 — Connection smoke test

We already have the SDK imported. Now we validate that the **Azure OpenAI v1 endpoint + API key work** before doing any real embedding.

### The endpoint we use

```
https://leonobitech.openai.azure.com/openai/v1/
```

Three important details:

1. **Subdomain `.openai.azure.com`** (not `.services.ai.azure.com`) — the Azure OpenAI endpoint lives in its own subdomain of the Foundry resource. Both coexist: `.services.ai.azure.com` is for Foundry Models / Agents; `.openai.azure.com` is for classic Azure OpenAI.
2. **Path `/openai/v1/`** with trailing slash — the `openai` SDK expects the URL ending in `/`. Without the slash, the SDK may build the final path incorrectly (concatenating `/embeddings` directly without a separator).
3. **No api-version in query** — the "v1" version is already in the path. The API is stable and the SDK handles it internally. (Different from the classic API `/openai/deployments/<name>/embeddings?api-version=2024-10-21` which does need an explicit version.)

### Auth

API key directly as `api_key=<key>` in the client constructor. Official quote from the docs:

> *"The Azure OpenAI embeddings API does not currently support Microsoft Entra ID with the v1 API. Use API key authentication."*

No `DefaultAzureCredential`, no `az login`, no token providers. Just the key from `.env`.

### How we validate it

We make an `embeddings.create` call with a minimal input (`"ping"`). If it responds without throwing an exception and returns a vector with the expected dimension (1536 for `text-embedding-3-small`), the setup works.

If it fails:

| Error | Probable cause |
|-------|----------------|
| 401 / 403 | Invalid API key |
| 404 | Endpoint without `/openai/v1/` or wrong base URL |
| 400 | `model` doesn't match the deployment name in the portal |

In [ ]:
# Connection smoke test — validates that URL + API key respond
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    base_url=os.getenv("AZURE_FOUNDRY_ENDPOINT"),
)

# Minimal call — without showing the vector, just validating that the API responds
response = client.embeddings.create(
    input=["ping"],
    model=os.getenv("AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT"),
)

vec = response.data[0].embedding

print(f"✅ Connection OK")
print(f"   base_url       : {client.base_url}")
print(f"   model          : {response.model}")
print(f"   vector_dim     : {len(vec)}")
print(f"   prompt_tokens  : {response.usage.prompt_tokens}")
print(f"   total_tokens   : {response.usage.total_tokens}")

## Sub-block 5 — Generate the first real embedding

We already validated the connection. Now we generate a "real" embedding with a specific input (`"smoke test from Foundry"`) to understand the vector shape, dimensions, and the raw output from the model.

### What is an embedding?

A **vector of floating-point numbers** that represents the semantic meaning of a text. For `text-embedding-3-small` the vector has **1536 dimensions** (numbers between -1 and 1 approximately).

The magic: two **semantically similar** texts produce vectors with **cosine close to 1**; unrelated texts produce cosine close to 0. That's what enables semantic search.

### The call

```python
client.embeddings.create(
    input="smoke test from Foundry",   # the text to vectorize
    model="text-embedding-3-small",     # = deployment name in Foundry
)
```

⚠️ **Gotcha**: the `model` you pass is the **Foundry deployment name**, NOT the OpenAI name. If your deployment has a different name in the portal (`my-embedder-prod`, etc.), use that one. They match when you deployed with the default name.

### The response

Structure of the object returned by the SDK:

```
CreateEmbeddingResponse(
    object='list',
    model='text-embedding-3-small',
    data=[
        Embedding(
            object='embedding',
            index=0,
            embedding=[0.0023, -0.0090, ...]   # 1536 floats
        )
    ],
    usage=Usage(prompt_tokens=4, total_tokens=4)
)
```

`response.data[0].embedding` is the list of 1536 floats. `response.usage` tells you how many tokens it consumed (useful for cost tracking in the next sub-block).

In [ ]:
# Generate embedding of the string "smoke test from Foundry"
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("AZURE_FOUNDRY_API_KEY"),
    base_url=os.getenv("AZURE_FOUNDRY_ENDPOINT"),
)

response = client.embeddings.create(
    input="smoke test from Foundry",
    model=os.getenv("AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT"),
)

vec = response.data[0].embedding

print(f"✅ Embedding generated")
print(f"   input          : 'smoke test from Foundry'")
print(f"   model          : {response.model}")
print(f"   vector_dim     : {len(vec)}")
print(f"   first_5_values : {vec[:5]}")
print(f"   tokens         : {response.usage.total_tokens}")

## Sub-block 6 — Reusable wrapper `FoundryEmbedder`

Final implementation in [`src/embeddings/foundry_client.py`](../src/embeddings/foundry_client.py).

*(pending — to be filled in the learning chat)*

## Sub-block 7 — Basic cost tracking

*(pending — to be filled in the learning chat)*